# TikTok 文字起こし・分析ツール

## 使い方（やることは 3 つだけ）
1. すぐ下の **▶（再生ボタン）を 1 回押す**
2. 途中で Google の画面が出たら、**自分のアカウントを選んで「許可」**（「続行」と出たら続行）
3. 数分待つと、下に **リンクとパスワード** が出るので、リンクを開いてログイン

※ ツールを使っている間、▶ のボタンはくるくる回ったままになります。それで正常です。このタブは閉じないでください。

In [ ]:
# @title ▶ ここを押すとツールが起動します
USE_SHEETS = False  # @param {type:"boolean"}
import os, sys, subprocess, secrets

print("① 準備中…（1〜3 分）")
if subprocess.run("nvidia-smi -L", shell=True, capture_output=True).returncode != 0:
    print("   ※ GPU なしで動かします（文字起こしが遅くなります）")
subprocess.run("rm -rf /content/TiTok- && git clone -q --depth 1 -b claude/beautiful-cori-0ern9t https://github.com/Daiki9615/TiTok-.git /content/TiTok-", shell=True, check=True)
subprocess.run(f"{sys.executable} -m pip install -q -U -r /content/TiTok-/requirements.txt", shell=True, check=True)
sys.path.insert(0, "/content/TiTok-")

print("② Google ドライブにつなぎます（許可の画面が出たら「許可」を押してください）")
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/TikTok文字起こし"
os.makedirs(DATA_DIR, exist_ok=True)

sheets = None
if USE_SHEETS:
    from google.colab import auth
    auth.authenticate_user()
    import google.auth, gspread
    from tiktok_tool.storage import SheetsMirror
    creds, _ = google.auth.default()
    sheets = SheetsMirror(gspread.authorize(creds), "TikTok文字起こしデータ")

from tiktok_tool.storage import DataStore
from tiktok_tool.fetcher import YtDlpFetcher
from tiktok_tool.asr import WhisperTranscriber
from tiktok_tool.app import launch

password = secrets.token_urlsafe(6)
print("")
print("③ 起動します。下に出る https://〇〇.gradio.live のリンクを開いてください")
print("   ユーザー名: user")
print("   パスワード:", password)
print("   （データの保存場所: マイドライブ / TikTok文字起こし）")
print("")
launch(DataStore(DATA_DIR), YtDlpFetcher(), WhisperTranscriber(), sheets, password=password, share=True, debug=True)

### 終わるとき
上のメニュー「ランタイム」→「ランタイムを接続解除して削除」

### スプレッドシートにも一覧を書き出したいとき
上の `USE_SHEETS` にチェックを入れてから ▶ を押す（Google の許可画面がもう 1 回出ます）